# Benchmark modelos de odio en español (Colab)

Esta notebook aplica clasificadores HuggingFace de odio en español a un CSV ya subido al entorno.

**Antes de ejecutar:**
1. Subir manualmente el CSV en *Archivos* (panel izquierdo de Colab) o arrastrarlo
2. Seleccionar entorno GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*
3. Ajustar `INPUT_CSV` en la celda de configuración si el nombre de archivo difiere

## 1. Setup

In [ ]:
import torch

device = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if device == 0 else "CPU"
print(f"Dispositivo: {device_name}")

## 2. Instalación

Descomenta si hace falta.

In [ ]:
# !pip install -q pandas transformers torch tqdm scikit-learn accelerate

## 3. Imports

In [ ]:
import re
from typing import Dict, Any, Optional

import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 4. Configuración

Ajusta estas variables según tu archivo.

In [ ]:
INPUT_CSV   = 'tweets_clasificados.csv'  # nombre del CSV subido al entorno
OUTPUT_CSV  = 'DNU_modelos.csv'

TEXT_COLUMN = 'text'   # columna con el texto a clasificar
GOLD_COLUMN = None     # columna con etiquetas de referencia; None si no hay gold
MAX_ROWS    = None     # None = procesar todo el archivo

## 5. Cargar datos

In [ ]:
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f'Filas originales: {len(df)}')
print('Columnas disponibles:')
print(list(df.columns))
df.head(3)

In [ ]:
text_col = TEXT_COLUMN
work_df = df.copy().head(MAX_ROWS).reset_index(drop=True)
print('Columna de texto:', text_col)
print('Filas a procesar:', len(work_df))

## 6. Modelos a comparar

- `cardiffnlp/twitter-xlm-roberta-base-hate-spanish` — orientado a hate speech en español en redes sociales.
- `JonatanGk/roberta-base-bne-finetuned-hate-speech-offensive-spanish` — clasifica hate/offensive en español.
- `delarosajav95/HateSpeech-BETO-cased-v2` — BETO fine-tuneado para detección de odio en español.

Poner `False` en `USE_MODELS` para desactivar alguno.

In [ ]:
USE_MODELS = {
    'cardiff_xlm_hate_es':     True,
    'bne_hate_offensive_es':   True,
    'beto_hate_v2_es':         True,
}

HF_MODEL_NAMES = {
    'cardiff_xlm_hate_es':   'cardiffnlp/twitter-xlm-roberta-base-hate-spanish',
    'bne_hate_offensive_es': 'JonatanGk/roberta-base-bne-finetuned-hate-speech-offensive-spanish',
    'beto_hate_v2_es':       'delarosajav95/HateSpeech-BETO-cased-v2',
}

## 7. Cargar pipelines

In [ ]:
loaded_models: Dict[str, Any] = {}
load_errors:   Dict[str, str] = {}

for key, model_name in HF_MODEL_NAMES.items():
    if USE_MODELS.get(key):
        try:
            loaded_models[key] = pipeline(
                'text-classification',
                model=model_name,
                tokenizer=model_name,
                truncation=True,
                device=device,
            )
            print(f'OK {key}: {model_name}')
        except Exception as e:
            load_errors[key] = str(e)
            print(f'ERROR {key}:', e)

print('\nErrores de carga:', load_errors)

## 8. Funciones auxiliares

Convierte las etiquetas de cada modelo a un esquema binario común:
- `1` = odio / ofensivo
- `0` = no odio

In [ ]:
NEGATIVE_TOKENS = {
    'non_hateful', 'non-hateful', 'not_hate', 'not-hate', 'no_hate', 'no-hate',
    'normal', 'neutral', 'none', 'clean', 'non-offensive', 'not offensive',
    'label_0', '0', 'neither',
}
POSITIVE_HINTS = {'hate', 'hateful', 'offensive', 'toxic', 'abusive', 'hs', 'aggressive', 'label_1', '1'}


def normalize_label_text(x: Any) -> str:
    if x is None:
        return ''
    s = str(x).strip().lower()
    return re.sub(r'\s+', ' ', s)


def binary_from_label(label: Any, score: Optional[float] = None) -> Optional[int]:
    s = normalize_label_text(label)
    if s in NEGATIVE_TOKENS:
        return 0
    for token in POSITIVE_HINTS:
        if token in s and s not in NEGATIVE_TOKENS:
            return 1
    return None


def run_hf_pipeline(model_key: str, text: str) -> Dict[str, Any]:
    clf = loaded_models[model_key]
    pred = clf(text)
    if isinstance(pred, list) and len(pred) > 0 and isinstance(pred[0], dict):
        best  = pred[0]
        label = best.get('label')
        score = best.get('score')
    else:
        label = str(pred)
        score = None
    return {'raw_label': label, 'score': score, 'binary_pred': binary_from_label(label, score)}

## 9. Prueba rápida con la primera fila

In [ ]:
sample_text = str(work_df[text_col].iloc[0])
print(sample_text)

test_outputs = {}
for model_key in loaded_models:
    try:
        test_outputs[model_key] = run_hf_pipeline(model_key, sample_text)
    except Exception as e:
        test_outputs[model_key] = {'error': str(e)}

test_outputs

## 10. Correr todos los modelos

Usa batching nativo del pipeline. Bajar `BATCH_SIZE` a 32 si hay OOM.

In [ ]:
BATCH_SIZE = 64

texts = work_df[text_col].fillna('').tolist()
n     = len(texts)

# Inicializar columnas
for model_key in loaded_models:
    for suffix in ['raw_label', 'score', 'binary_pred', 'error']:
        work_df[f'{model_key}__{suffix}'] = None

# Procesar cada modelo con pipeline batching
for model_key, clf in loaded_models.items():
    raw_labels, scores, binary_preds = [], [], []

    for out in tqdm(clf(texts, batch_size=BATCH_SIZE), total=n, desc=model_key):
        label = out.get('label')
        score = out.get('score')
        raw_labels.append(label)
        scores.append(score)
        binary_preds.append(binary_from_label(label, score))

    work_df[f'{model_key}__raw_label']   = raw_labels
    work_df[f'{model_key}__score']       = scores
    work_df[f'{model_key}__binary_pred'] = binary_preds

work_df.to_csv(OUTPUT_CSV, index=False)
print(f'\nResultados guardados en {OUTPUT_CSV}')

## 11. Vista rápida de resultados

In [ ]:
cols_to_show = [text_col]
if GOLD_COLUMN is not None and GOLD_COLUMN in work_df.columns:
    cols_to_show.append(GOLD_COLUMN)
for model_key in loaded_models:
    cols_to_show += [f'{model_key}__raw_label', f'{model_key}__binary_pred']
work_df[cols_to_show].head(10)

## 12. Comparación entre modelos

Cuántos casos positivos marca cada uno.

In [ ]:
summary_rows = []
for model_key in loaded_models:
    pred_col = f'{model_key}__binary_pred'
    valid = pd.to_numeric(work_df[pred_col], errors='coerce')
    summary_rows.append({
        'model':         model_key,
        'n_valid_preds': int(valid.notna().sum()),
        'n_pred_hate':   int((valid == 1).sum()),
        'hate_rate':     float((valid == 1).mean()) if valid.notna().any() else None,
    })
summary_df = pd.DataFrame(summary_rows).sort_values('hate_rate', ascending=False)
summary_df

## 13. Acuerdo entre modelos

Correlación entre predicciones binarias.

In [ ]:
pred_cols   = [f'{m}__binary_pred' for m in loaded_models]
pred_matrix = work_df[pred_cols].apply(pd.to_numeric, errors='coerce')
pred_matrix.corr()

## 14. Evaluación contra etiqueta de referencia (opcional)

Esta celda solo corre si `GOLD_COLUMN` está definida y existe en el CSV.

In [ ]:
if GOLD_COLUMN is None or GOLD_COLUMN not in work_df.columns:
    print('GOLD_COLUMN no definida o no encontrada — salteando evaluación.')
else:
    gold_bin = pd.to_numeric(work_df[GOLD_COLUMN], errors='coerce')
    valid_gold = gold_bin.notna()
    print(f'Filas con gold label: {valid_gold.sum()}')

    for model_key in loaded_models:
        pred_col = f'{model_key}__binary_pred'
        preds    = pd.to_numeric(work_df[pred_col], errors='coerce')
        mask     = valid_gold & preds.notna()
        y_true   = gold_bin[mask].astype(int)
        y_pred   = preds[mask].astype(int)

        print('\n' + '=' * 60)
        print(f'Modelo: {model_key}')
        print(f'Predicciones válidas: {mask.sum()} / {len(work_df)}')
        print('=' * 60)
        if len(y_true) == 0:
            print('Sin predicciones válidas.')
            continue
        print('Accuracy:', round(accuracy_score(y_true, y_pred), 4))
        print('Confusion matrix:')
        print(confusion_matrix(y_true, y_pred))
        print()
        print(classification_report(y_true, y_pred, target_names=['NO ODIO', 'ODIO'], digits=4))

## 15. Descargar resultado

In [ ]:
from google.colab import files
files.download(OUTPUT_CSV)